# WLASL landmark accuracy chase v2 (Colab T4)

Trains **velocity Transformer multi-seed + Pose-TGCN** on `wlasl_landmarks_v3` (50 glosses).

**v2 fix:** use the proven Transformer size (`d_model=128`, 3 layers) — the larger 192/4 model failed on Colab (~5% val). Keep the stronger TGCN settings (3 seeds, hidden 128).

**Runtime → Change runtime type → T4 GPU** before running.

When finished, download `outputs/chase_colab_v2_results.zip` and bring it back to the laptop project.


In [ ]:
# @title 1) Setup paths
from pathlib import Path

# Upload wlasl_colab_chase.zip to Colab Files, or point at Drive:
ZIP_PATH = Path('/content/wlasl_colab_chase.zip')
# ZIP_PATH = Path('/content/drive/MyDrive/wlasl_colab_chase.zip')

USE_DRIVE = False  # True => also copy results zip to Drive
DRIVE_OUT = Path('/content/drive/MyDrive/wlasl_chase_outputs')

WORK = Path('/content/wlasl_colab_chase')
OUT = WORK / 'outputs' / 'chase_colab_v2'
print('ZIP exists:', ZIP_PATH.exists(), ZIP_PATH)
print('OUT:', OUT)


In [ ]:
# @title 2) Optional: mount Drive
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_OUT.mkdir(parents=True, exist_ok=True)
    print("Drive ready:", DRIVE_OUT)

In [ ]:
# @title 3) Unzip package
import zipfile
import shutil
from pathlib import Path

CONTENT = Path('/content')
WORK = CONTENT / 'wlasl_colab_chase'

if (WORK / 'data' / 'processed' / 'wlasl_landmarks_v3' / 'metadata.json').exists():
    print('Already unpacked:', WORK)
else:
    assert ZIP_PATH.exists(), f'Upload zip first: {ZIP_PATH}'
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall(CONTENT)
    assert (WORK / 'data' / 'processed' / 'wlasl_landmarks_v3' / 'metadata.json').exists(), WORK
    print('Unpacked to', WORK)

OUT = WORK / 'outputs' / 'chase_colab_v2'
OUT.mkdir(parents=True, exist_ok=True)
!ls -la {WORK}
!ls {WORK}/data/processed/wlasl_landmarks_v3 | head


In [ ]:
# @title 4) Install deps + verify GPU
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "joblib", "scikit-learn", "numpy"])

import torch
print("torch", torch.__version__)
print("cuda_available", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU — Runtime → Change runtime type → T4 GPU")

In [ ]:
# @title 5) Train chase ensemble v2 (proven Transformer size)
import os, sys, json, time
from pathlib import Path

os.chdir(WORK)
sys.path.insert(0, str(WORK))

from modules.recognition.wlasl_boost_trainer import train_boosted_landmark_models

OUT.mkdir(parents=True, exist_ok=True)
print('cuda?', __import__('torch').cuda.is_available())
t0 = time.time()
report = train_boosted_landmark_models(
    processed_dir=str(WORK / 'data' / 'processed' / 'wlasl_landmarks_v3'),
    output_dir=str(OUT),
    epochs=140,
    aug_copies=8,
    n_seeds=5,
    train_tgcn_flag=True,
    # Proven Transformer (v1 Colab 192/4 collapsed to ~5%)
    d_model=128,
    nhead=4,
    layers=3,
    n_tgcn_seeds=3,
    n_tta=9,
    transformer_lr=8e-4,
    # Keep stronger TGCN from v1
    tgcn_hidden=128,
    tgcn_stages=14,
)
elapsed = time.time() - t0
print(f'\nDone in {elapsed/60:.1f} min')
print(json.dumps({
    'val_accuracy': report['val_accuracy'],
    'baseline_to_beat': 0.6068,
    'colab_v1_tgcn_only': 0.6239,
    'transformer_ensemble': report['transformer_ensemble_val_accuracy'],
    'tgcn': report['tgcn_val_accuracy'],
    'hgb': report['hgb_val_accuracy'],
    'blend_weights': report['blend_weights'],
    'seed_val_accuracies': report['seed_val_accuracies'],
    'accuracy_met_85': report['accuracy_met'],
}, indent=2))


In [ ]:
# @title 6) Package results for download
import json, shutil, zipfile
from pathlib import Path

summary = {
    'run': 'chase_colab_v2',
    'd_model': 128,
    'layers': 3,
    'val_accuracy': report['val_accuracy'],
    'num_classes': report['num_classes'],
    'n_val': report['n_val'],
    'blend_weights': report['blend_weights'],
    'transformer_ensemble_val_accuracy': report['transformer_ensemble_val_accuracy'],
    'tgcn_val_accuracy': report['tgcn_val_accuracy'],
    'hgb_val_accuracy': report['hgb_val_accuracy'],
    'seed_val_accuracies': report['seed_val_accuracies'],
    'baseline_to_beat': 0.6068376068376068,
    'colab_v1_tgcn_only': 0.6239316239316239,
    'source': 'colab_t4_chase_v2',
}
(OUT / 'colab_summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')

result_zip = WORK / 'outputs' / 'chase_colab_v2_results.zip'
result_zip.parent.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(result_zip, 'w', zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(OUT.rglob('*')):
        if p.is_file():
            zf.write(p, arcname=str(Path('chase_colab_v2') / p.relative_to(OUT)))

print('Wrote', result_zip, 'size_mb', round(result_zip.stat().st_size / 1e6, 2))
!ls -lh {OUT}

if USE_DRIVE:
    dest = DRIVE_OUT / 'chase_colab_v2_results.zip'
    DRIVE_OUT.mkdir(parents=True, exist_ok=True)
    shutil.copy2(result_zip, dest)
    print('Copied to Drive:', dest)

try:
    from google.colab import files
    files.download(str(result_zip))
except Exception as e:
    print('Auto-download skipped:', e)
    print('Download manually from', result_zip)
